### Install the necessarey Modules

In [0]:
!pip install psycopg2-binary
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Connectivity Check

In [0]:
import psycopg2

conn = psycopg2.connect(
    host="{Give your DB Host Name here}",   # e.g. db.example.com
    port=5432,
    database="postgres",
    user="snowflake_admin",
    password="{Give Your DB Password here}",
    sslmode="require"
)

cur = conn.cursor()
cur.execute("SELECT version();")
print(cur.fetchone())

('PostgreSQL 18.3 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 11.5.0 20240719 (Red Hat 11.5.0-11), 64-bit',)


### Install pgvector and create the Semantic Cache Table(One Time Activity)

In [0]:
cur.execute("CREATE EXTENSION if not exists vector;")
cur.execute("DROP TABLE IF EXISTS cache;")
cur.execute("""
        CREATE  TABLE cache (
            id SERIAL PRIMARY KEY,
            question TEXT NOT NULL,
            answer TEXT NOT NULL,
            embedding vector(1024),
            created_at TIMESTAMP DEFAULT NOW(),
            ttl_seconds INT DEFAULT 3600
        );
    """)

conn.commit()
cur.close()
conn.close()

### Create a Reusable Database Connection Function:

In [0]:
def get_connection():
    return psycopg2.connect(
    host="{Give your DB Host Name here}",   # e.g. db.example.com
    port=5432,
    database="postgres",
    user="snowflake_admin",
    password="{Give Your DB Password here}",
    sslmode="require"
)

In [0]:
CHAT_MODEL = 'databricks-meta-llama-3-3-70b-instruct'
CACHE_TTL_SECONDS = 300
EMBEDDING_MODEL = 'databricks-bge-large-en'
SIMILARITY_THRESHOLD = 0.90

### Create Function for Vector Embeddings Using Databricks Mosaic AI

In [0]:
def get_embedding(text: str):

    query = f"""
    SELECT ai_query(
        '{EMBEDDING_MODEL}',
        request => '{text.replace("'", "''")}'
    ) as embedding_vector
    """

    embedding = spark.sql(query).collect()[0]["embedding_vector"]

    return embedding

### Create Function to search for similar questions in the Semantic Cache Table

In [0]:
def find_similar_cached_response(embedding):

    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
        SELECT
            question,
            answer,
            1 - (embedding <=> %s::vector) as similarity
        FROM cache
        ORDER BY embedding <=> %s::vector
        LIMIT 1;
    """, (embedding, embedding))

    row = cur.fetchone()
    print("Best Matching data from Cache: ",row)

    cur.close()
    conn.close()

    if row and row[2] >= SIMILARITY_THRESHOLD:
        print(f"Cache HIT ({row[2]:.4f})")
        return row[1]

    print("Cache MISS")

    return None

### Create the function to generate a Fresh Response on Cache Miss

In [0]:
def generate_ai_response(question: str):

    query = f"""
    SELECT ai_query(
        '{CHAT_MODEL}',
        CONCAT(
            'You are a helpful assistant. Answer concisely. User question: ',
            '{question.replace("'", "''")}'
        )
    ) as llm_response
    """

    response = spark.sql(query).collect()[0]["llm_response"]

    return response

### Persist New Responses Back to the Semantic Cache for Cache Miss case

In [0]:
def persist_response_to_cache(question, answer, embedding):

    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
        INSERT INTO cache (
            question,
            answer,
            embedding,
            ttl_seconds
        )
        VALUES (%s,%s,%s::vector,%s)
    """, (
        question,
        answer,
        embedding,
        CACHE_TTL_SECONDS
    ))

    conn.commit()

    cur.close()
    conn.close()

### Orchestrating the End-to-End Semantic Caching Flow

In [0]:
import time

def ask(question):

    # Start timer to measure end-to-end latency
    start = time.time()

    print(f"\nUser Question: {question}")

    # ---------------------------------------------------
    # Step 1:
    # Convert user question into vector embedding
    # Example:
    # "What is Apache Spark?" -> [0.12, -0.45, ...]
    # ---------------------------------------------------
    print("Generating embedding...")
    embedding = get_embedding(question)

    # ---------------------------------------------------
    # Step 2:
    # Search semantic cache for similar questions
    # If similarity > threshold, cached response is returned
    # ---------------------------------------------------
    print("Searching semantic cache...")
    cached_answer = find_similar_cached_response(embedding)

    # ---------------------------------------------------
    # Step 3:
    # Cache HIT → No LLM call needed
    # Directly return cached response
    # This saves both cost and latency
    # ---------------------------------------------------
    if cached_answer:

        latency = (time.time() - start) * 1000

        print("Serving response from cache")
        print(f"End-to-End Latency: {latency:.0f}ms")

        return cached_answer

    # ---------------------------------------------------
    # Step 4:
    # Cache MISS → Need fresh LLM generation
    # Usually happens for new/distinct questions
    # ---------------------------------------------------
    print("No semantic match found, invoking LLM...")
    answer = generate_ai_response(question)

    # ---------------------------------------------------
    # Step 5:
    # Store new question + answer + embedding
    # So future similar questions become cache hits
    # ---------------------------------------------------
    print("Storing response in semantic cache...")
    persist_response_to_cache(
        question,
        answer,
        embedding
    )

    # ---------------------------------------------------
    # Step 6:
    # Track overall latency after full pipeline execution
    # ---------------------------------------------------
    latency = (time.time() - start) * 1000

    print("Serving fresh LLM response")
    print(f"End-to-End Latency: {latency:.0f}ms")

    return answer

### Asking a Question without Semantic Cache

In [0]:
import time

question = "What is Apache Spark?"

# Start timer
start = time.time()

# Direct LLM call (no cache involved)
response = generate_ai_response(question)

# End timer
end = time.time()

# Total latency in milliseconds
latency_ms = (end - start) * 1000

print("Response:")
print(response)

print(f"\nTotal Latency: {latency_ms:.0f}ms")

Response:
Apache Spark is an open-source, unified analytics engine for large-scale data processing, providing high-performance, in-memory computing for batch, interactive, and real-time data processing.

Total Latency: 10080ms


In [0]:
print(ask("What is Apache Spark?"))


User Question: What is Apache Spark?
Generating embedding...
Searching semantic cache...
Best Matching data from Cache:  None
Cache MISS
No semantic match found, invoking LLM...
Storing response in semantic cache...
Serving fresh LLM response
End-to-End Latency: 2417ms
Apache Spark is an open-source, unified analytics engine for large-scale data processing, providing high-performance, in-memory computing for batch, interactive, and real-time data processing.


In [0]:
import time

question = "Can you explain Apache Spark?"

# Start timer
start = time.time()

# Direct LLM call (no cache involved)
response = generate_ai_response(question)

# End timer
end = time.time()

# Total latency in milliseconds
latency_ms = (end - start) * 1000

print("Response:")
print(response)

print(f"\nTotal Latency: {latency_ms:.0f}ms")

Response:
Apache Spark is an open-source, unified analytics engine for large-scale data processing. It provides high-level APIs in Java, Python, and Scala for tasks like data ingestion, processing, and analytics, supporting batch, real-time, and machine learning workloads.

Total Latency: 1582ms


In [0]:
print(ask("Can you explain Apache Spark?"))


User Question: Can you explain Apache Spark?
Generating embedding...
Searching semantic cache...
Best Matching data from Cache:  ('What is Apache Spark?', 'Apache Spark is an open-source, unified analytics engine for large-scale data processing, providing high-performance, in-memory computing for batch, interactive, and real-time data processing.', 0.9118453780416437)
Cache HIT (0.9118)
Serving response from cache
End-to-End Latency: 769ms
Apache Spark is an open-source, unified analytics engine for large-scale data processing, providing high-performance, in-memory computing for batch, interactive, and real-time data processing.


### Query Snowflake Postgres for intuition

In [0]:
import pandas as pd

# Create connection
conn = get_connection()

# Read cache table into pandas dataframe
df = pd.read_sql("""
    SELECT *
    FROM cache
    ORDER BY created_at DESC
""", conn)

# Close connection
conn.close()

# Display dataframe
display(df)

/home/spark-75acccb1-9a2b-40cd-a275-55/.ipykernel/6222/command-5443438533056856-1545201167:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


id question answer embedding created_at ttl_seconds 1 What is Apache Spark? Apache Spark is an open-source, unified analytics engine for large-scale data processing, providing high-performance, in-memory computing for batch, interactive, and real-time data processing. [0.018569946,-0.014030457,-0.057647705,0.0034484863,0.008575439,-0.02168274,-0.024734497,-0.004714966,0.013626099,0.050323486,-0.027496338,-0.014724731,0.054748535,-0.05380249,-0.010253906,-0.016189575,-0.01876831,-0.017181396,-0.05117798,0.017868042,0.0042877197,0.028533936,-0.055480957,-0.037750244,-0.0012273788,0.020187378,-0.04675293,0.01586914,0.09375,0.019485474,-0.044708252,-0.01235199,-0.0063476562,-0.029190063,0.043273926,-0.02532959,0.049468994,0.032409668,-0.06011963,-0.012512207,0.021896362,-0.0099487305,0.013473511,-0.04043579,-0.07763672,-0.04083252,0.003168106,-0.033721924,-0.012931824,-0.040985107,-0.039886475,0.0231781,0.0033359528,-0.0020122528,6.2286854e-05,0.010795593,-0.020568848,-0.0028896332,-0.058410645,0.005958557,-0.009468079,0.04421997,0.017700195,-0.060699463,-0.0022411346,0.050109863,-0.007118225,-0.00415802,0.0087509155,-0.026992798,-0.05117798,0.026489258,-0.009674072,-0.023834229,-0.026733398,0.027191162,0.010467529,-0.020187378,0.007698059,0.04815674,-0.010299683,0.017959595,-0.021850586,0.029434204,-0.037719727,-0.04559326,0.077819824,-0.008895874,-0.0029582977,0.020355225,-0.020950317,0.02458191,0.03475952,0.01676941,0.006046295,0.051971436,-0.037231445,0.04711914,0.025314331,0.048828125,0.02520752,0.023803711,-0.045135498,0.016571045,0.0016994476,-0.005672455,0.017501831,0.012550354,-0.042816162,-0.041412354,-0.0040359497,-0.020950317,-0.019699097,-0.011047363,0.0049591064,0.009178162,-0.013412476,0.02458191,0.009841919,-0.057495117,-0.010353088,0.0068740845,0.020019531,-0.02168274,-0.041778564,-0.073913574,-0.0121536255,0.04711914,0.013061523,0.0035591125,0.030975342,-0.02079773,-0.061462402,0.01927185,0.0016727448,-0.022277832,0.031982422,0.037322998,0.04763794,-0.016159058,-0.028335571,0.021057129,-0.03213501,0.07824707,-0.010925293,0.03878784,0.0032367706,-0.020629883,-0.07324219,0.053466797,-0.025253296,-0.02243042,0.04888916,0.021438599,-0.013702393,0.032592773,-0.013160706,0.024627686,0.038513184,-0.06640625,0.0004310608,0.011672974,-0.021316528,-0.0029525757,-0.023757935,0.02798462,0.010848999,-0.019348145,-0.034362793,0.011711121,0.0040283203,0.0012760162,-0.040008545,0.000374794,0.06298828,-0.012207031,-0.0023288727,-0.027770996,0.001698494,0.033966064,4.5776367e-05,-0.011909485,-0.029205322,0.0044403076,-0.024917603,0.041625977,0.015960693,0.013755798,0.009941101,-0.019592285,0.010414124,0.08874512,-0.051239014,0.024032593,-0.004295349,-0.008651733,-0.031982422,0.0033550262,-0.011253357,-0.039611816,0.00070667267,0.0119018555,0.009162903,-0.03189087,-0.0046844482,-0.032073975,0.008552551,0.005634308,-0.04949951,-0.03363037,0.058624268,0.033935547,-0.0040359497,0.0010223389,0.0446167,-0.022964478,-0.0053901672,0.013969421,0.012496948,-0.0029754639,-0.010932922,0.015808105,0.033691406,0.060333252,-0.019821167,-0.012840271,-0.029693604,-0.005142212,-0.03213501,-0.040618896,0.035125732,0.04296875,-0.011260986,0.016616821,-0.0014734268,-0.020904541,-0.009346008,0.045776367,-0.040283203,0.0043792725,-0.0038204193,0.014480591,0.0008611679,-0.0014257431,0.04901123,0.007091522,0.023468018,-0.027786255,-0.0042381287,0.04333496,-0.0206604,0.02281189,0.027267456,0.009437561,-0.021362305,-0.01309967,-0.038482666,0.004486084,-0.021148682,-0.040985107,-0.0045547485,0.018585205,-0.00033688545,0.0035972595,0.043945312,0.04232788,0.009971619,-0.0026893616,-0.002729416,-0.003080368,-0.023757935,-0.0637207,-0.0016880035,-0.010940552,-0.062927246,0.022033691,-0.011421204,-0.06726074,0.041046143,-0.0020866394,0.0033187866,-0.019561768,-0.0022602081,-0.017684937,0.03237915,-0.010910034,-0.026443481,-0.0046195984,-0.03277588,-0.0037231445,-0.012077332,0.0015850067,0.023269653,0.026031494,0.009597778,-0.00024008751,-0.015792847,0.019180298,0

### Managing Cache Expiration

In [0]:
def cleanup_expired_cache_entries():
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        DELETE FROM cache
        WHERE created_at < NOW() - INTERVAL '1 second' * ttl_seconds;
    """)
    deleted = cur.rowcount
    conn.commit()
    cur.close()
    conn.close()
    print(f"Evicted {deleted} expired entries.")

In [0]:
cleanup_expired_cache_entries()

Evicted 1 expired entries.
